# ==============================================================================
# INTRODU??O AO APRENDIZADO DE M?QUINA (EEL891)
# TRABALHO 2: PREVIS?O DE PRE?OS DE IM?VEIS
# ALUNO: MURILO ARAUJO GARCIA
# KAGGLE ID: muriloaraujogarcia (ou n?mero da conta: 25727406)
# DRE: 124179602
# EMAIL: MURILOGARCIA@UFRJ.BR
# ==============================================================================

## 1. INTRODU??O

O mercado imobili?rio ? um pilar fundamental da economia, caracterizado por sua complexidade e pela vasta quantidade de fatores que influenciam a precifica??o de um im?vel. A habilidade de prever com acur?cia o valor de um bem imobili?rio ? de grande interesse para investidores, corretores e compradores.

Este trabalho se prop?e a desenvolver um modelo de aprendizado de m?quina robusto para a tarefa de previs?o de pre?os de im?veis, utilizando um conjunto de dados que descreve diversas caracter?sticas de apartamentos e casas. O objetivo ? aplicar e avaliar t?cnicas de pr?-processamento, engenharia de atributos e modelagem preditiva para alcan?ar o melhor desempenho poss?vel na m?trica de **Root Mean Squared Logarithmic Error (RMSLE)**, conforme a competi??o proposta no Kaggle.

A seguir, ser? detalhado todo o processo de desenvolvimento, desde a an?lise explorat?ria inicial dos dados at? a constru??o e treinamento do modelo final.

In [ ]:
# ==============================================================================
# BLOCO 1: CONFIGURA??O E IMPORTA??O DE BIBLIOTECAS
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

import warnings
warnings.filterwarnings('ignore')

# Configura??es de visualiza??o para os gr?ficos
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("? Bloco 1 conclu?do: Bibliotecas importadas!")

In [ ]:
# ==============================================================================
# BLOCO 2.1: CARREGAMENTO DOS DADOS
# ==============================================================================
# Carregando os dados
df_train_raw = pd.read_csv('conjunto_de_treinamento.csv')
df_test_raw = pd.read_csv('conjunto_de_teste.csv')

# Guardando IDs para a submiss?o final
test_ids = df_test_raw['Id']

# Removendo a coluna Id para n?o ser usada no treinamento
df_train = df_train_raw.drop('Id', axis=1)
df_test = df_test_raw.drop('Id', axis=1)

print("Dados de treinamento carregados. Dimens?es:", df_train.shape)
print("Dados de teste carregados. Dimens?es:", df_test.shape)
df_train.head()

## 2. AN?LISE EXPLORAT?RIA E PR?-PROCESSAMENTO DOS DADOS

Antes de treinar qualquer modelo, ? crucial entender a natureza dos dados, suas distribui??es, correla??es e a presen?a de poss?veis anomalias. Esta etapa fundamenta todas as decis?es de pr?-processamento e engenharia de atributos.

### 2.1. An?lise da Vari?vel Alvo: `preco`

A vari?vel que desejamos prever (`preco`) ? a mais importante. Sua distribui??o nos informa sobre a escala dos valores e a necessidade de transforma??es. Um histograma ? a ferramenta ideal para essa an?lise.

A an?lise do histograma revela uma forte **assimetria positiva (right-skewed)**, com uma longa cauda de im?veis de alto valor. Essa distribui??o pode prejudicar modelos lineares e fazer com que o treinamento seja desproporcionalmente influenciado por outliers.

A m?trica da competi??o (RMSLE) utiliza o logaritmo dos pre?os, o que naturalmente penaliza erros percentuais de forma mais equilibrada. Portanto, aplicar uma **transforma??o logar?tmica** (`np.log1p`) na vari?vel alvo ? uma etapa estrat?gica que alinha nosso processo de treinamento com o objetivo da competi??o e, ao mesmo tempo, normaliza a distribui??o.

In [ ]:
# Aplicando a transforma??o logar?tmica e visualizando o resultado
y_log_transformed = np.log1p(df_train['preco'])

plt.figure(figsize=(14, 6))
sns.histplot(y_log_transformed, kde=True, bins=50, color='green')
plt.title('Distribui??o dos Pre?os dos Im?veis (Ap?s Transforma??o Logar?tmica)', fontsize=16)
plt.xlabel('Log(Pre?o + 1)', fontsize=12)
plt.ylabel('Frequ?ncia', fontsize=12)
plt.show()

print("Assimetria (Skewness) ap?s log:", y_log_transformed.skew())

A transforma??o logar?tmica ajuda a mitigar o efeito de outliers, mas a remo??o de valores claramente an?malos ainda ? uma boa pr?tica. Durante a an?lise, foram identificados im?veis com pre?os e ?reas ?teis desproporcionalmente altos, que poderiam representar erros de digita??o ou casos muito at?picos.

A estrat?gia adotada foi remover um outlier de pre?o extremo (acima de 30 milh?es) e, em seguida, filtrar os dados para manter apenas os im?veis que est?o dentro do **percentil 99** tanto para pre?o quanto para ?rea ?til. Essa abordagem remove as anomalias mais gritantes sem sacrificar uma quantidade significativa de dados de treinamento.

In [ ]:
# ==============================================================================
# BLOCO 2.2: LIMPEZA AVAN?ADA DOS DADOS
# ==============================================================================
print(f"Shape do treino antes da limpeza: {df_train.shape}")

# --- Tratamento de Outliers ---
df_train = df_train[df_train['preco'] < 30000000]
price_cap = df_train['preco'].quantile(0.99)
area_cap = df_train['area_util'].quantile(0.99)
df_train = df_train[df_train['preco'] <= price_cap]
df_train = df_train[df_train['area_util'] <= area_cap]

print(f"Shape do treino ap?s limpeza: {df_train.shape}")
print("\n? Outliers removidos com sucesso!")

### 2.2. An?lise de Correla??o entre Atributos Num?ricos

Para entender as rela??es lineares entre as vari?veis, um **heatmap de correla??o** ? extremamente ?til. Ele nos ajuda a identificar quais atributos t?m maior rela??o com o `preco` e se h? multicolinearidade (alta correla??o entre atributos preditores), o que poderia ser um problema para alguns modelos.

In [ ]:
# Selecionando apenas colunas num?ricas para o heatmap
numerical_cols = df_train.select_dtypes(include=np.number).columns
correlation_matrix = df_train[numerical_cols].corr()

plt.figure(figsize=(16, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=.5)
plt.title('Heatmap de Correla??o entre Atributos Num?ricos', fontsize=16)
plt.show()

O heatmap confirma algumas intui??es:
- **Forte Correla??o Positiva com o Pre?o:** Atributos como `area_util`, `quartos`, `suites` e `vagas` est?o, como esperado, fortemente correlacionados com o `preco`.
- **Multicolinearidade:** Existe uma alta correla??o entre `quartos` e `suites` (0.70), o que ? natural. Modelos baseados em ?rvores, como os que ser?o utilizados (LGBM, XGBoost, CatBoost), s?o inerentemente robustos a esse tipo de multicolinearidade, n?o sendo necess?rio remover nenhuma dessas vari?veis.

## 3. ENGENHARIA DE ATRIBUTOS

Esta ? talvez a etapa mais crucial para o sucesso de um modelo preditivo. O objetivo ? criar novos atributos a partir dos existentes para fornecer ao modelo informa??es mais ricas e contextuais que ele n?o conseguiria capturar diretamente.

A estrat?gia adotada foi a seguinte:

1.  **Cria??o de Features de Intera??o:** Combinar atributos para criar novas m?tricas, como `comodos_total` (soma de quartos e su?tes) e `area_por_quarto`.
2.  **Cria??o de Features de Propor??o:** Calcular a propor??o da ?rea ?til em rela??o ? ?rea total (`prop_area_util`), o que pode indicar a efici?ncia do aproveitamento do espa?o.
3.  **Target Encoding Manual (A "Dica do Professor"):** Esta foi a t?cnica de maior impacto. Em vez de simplesmente tratar `bairro` como uma categoria, foram criados dois novos atributos para cada im?vel:
    * `preco_medio_bairro`: O pre?o m?dio de todos os im?veis no mesmo bairro.
    * `preco_std_bairro`: O desvio padr?o dos pre?os no mesmo bairro.

Essa abordagem injeta diretamente no modelo a informa??o sobre o **contexto de mercado** de cada bairro, diferenciando bairros de alta e baixa valoriza??o de forma muito eficaz.

In [ ]:
# ==============================================================================
# BLOCO 3: ENGENHARIA DE FEATURES AVAN?ADA (A RECEITA VENCEDORA)
# ==============================================================================
def feature_engineering_avancada(df, training_df):
    """
    Aplica um conjunto de transforma??es e cria novas features robustas.
    Usa o df de treino como refer?ncia para os c?lculos de m?dia e desvio padr?o.
    """
    # Combina??es de features existentes
    df['comodos_total'] = df['quartos'] + df['suites']
    df['area_por_quarto'] = df['area_util'] / (df['quartos'] + 1)
    df['prop_area_util'] = df['area_util'] / (df['area_util'] + df['area_extra'] + 1)

    # A feature vencedora: Target Encoding Manual
    bairro_stats = training_df.groupby('bairro')['preco'].agg(['mean', 'std']).rename(columns={'mean': 'preco_medio_bairro', 'std': 'preco_std_bairro'})
    df = df.merge(bairro_stats, on='bairro', how='left')
    
    # Preenche NaNs que possam surgir no conjunto de teste com a m?dia geral do treino
    df['preco_medio_bairro'] = df['preco_medio_bairro'].fillna(training_df['preco'].mean())
    df['preco_std_bairro'] = df['preco_std_bairro'].fillna(training_df['preco'].std())
    
    df.fillna(0, inplace=True)
    return df

df_train_eng = feature_engineering_avancada(df_train, df_train)
df_test_eng = feature_engineering_avancada(df_test, df_train)

train_cols = df_train_eng.columns.drop('preco')
df_test_eng = df_test_eng.reindex(columns=train_cols, fill_value=0)

print("? Bloco 3 conclu?do: Engenharia de features avan?ada aplicada!")

## 4. MODELAGEM PREDITIVA

Com os dados devidamente tratados e enriquecidos, a pr?xima etapa ? a constru??o e o treinamento dos modelos preditivos.

### 4.1. Pipeline de Pr?-processamento Final

Para garantir que todas as transforma??es sejam aplicadas de forma consistente nos dados de treino e de teste, foi constru?da uma `Pipeline` utilizando o `ColumnTransformer` do Scikit-learn. Ela realiza as seguintes tarefas:

- **Para Atributos Num?ricos:** Imputa??o de valores faltantes com a **mediana** (robusto a outliers) e aplica??o de `StandardScaler` para padronizar a escala.
- **Para Atributos Categ?ricos:** Imputa??o de faltantes com o valor mais frequente e aplica??o de `OneHotEncoder` para transformar as categorias em formato num?rico.
- **Para o Atributo de Texto (`diferenciais`):** Utiliza??o do `TfidfVectorizer` para extrair as 100 palavras e bigramas mais relevantes, convertendo texto n?o estruturado em features num?ricas.

### 4.2. Escolha dos Modelos

Foram escolhidos tr?s modelos de **Gradient Boosting**, conhecidos por sua alta performance em competi??es de dados tabulares como o Kaggle:

1.  **LightGBM (LGBM):** Famoso por sua velocidade e efici?ncia, especialmente com grandes datasets. O objetivo de otimiza??o foi definido como `regression_l1` (Erro Absoluto M?dio - MAE), que ? mais robusto a outliers, alinhando-se com a estrat?gia de usar a mediana na imputa??o.
2.  **XGBoost:** Um dos modelos mais populares e vitoriosos em competi??es, conhecido por sua performance e regulariza??o.
3.  **CatBoost:** Especialmente poderoso para lidar com atributos categ?ricos, embora aqui tenha sido usado em conjunto com o One-Hot Encoding.

### 4.3. Estrat?gia de Ensemble

Em vez de depender de um ?nico modelo, foi utilizada uma t?cnica de **ensemble**, combinando as previs?es dos tr?s modelos. A combina??o foi feita atrav?s de uma **m?dia ponderada**, dando um peso maior (40% cada) para o LGBM e XGBoost, que historicamente performam melhor, e um peso menor (20%) para o CatBoost. Isso cria um "comit? de especialistas" que tende a ser mais robusto e preciso do que qualquer especialista individual.

In [ ]:
# ==============================================================================
# BLOCO 4: PREPROCESSAMENTO E PIPELINES
# ==============================================================================
X = df_train_eng.drop('preco', axis=1)
y_log = np.log1p(df_train_eng['preco'])
X_test = df_test_eng

numerical_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
text_feature = 'diferenciais'
if text_feature in categorical_features:
    categorical_features.remove(text_feature)

numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))])
text_transformer = Pipeline(steps=[('tfidf', TfidfVectorizer(max_features=100, stop_words=['e', 'de', 'a', 'o', 'que', 'para'], ngram_range=(1,2)))])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features),
        ('text', text_transformer, text_feature)
    ],
    remainder='passthrough'
)

print("\n? Bloco 4 conclu?do: Pipelines de pr?-processamento definidos!")

In [ ]:
# ==============================================================================
# BLOCO 5: DEFINI??O E TREINAMENTO DOS MODELOS
# ==============================================================================
lgbm_params = {'objective': 'regression_l1', 'metric': 'rmse', 'n_estimators': 2000, 'learning_rate': 0.01, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 1, 'lambda_l1': 0.1, 'lambda_l2': 0.1, 'num_leaves': 31, 'verbose': -1, 'n_jobs': -1, 'seed': 42}
xgb_params = {'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'n_estimators': 2000, 'learning_rate': 0.01, 'max_depth': 5, 'subsample': 0.7, 'colsample_bytree': 0.7, 'random_state': 42, 'n_jobs': -1}
cat_params = {'iterations': 2000, 'learning_rate': 0.05, 'depth': 6, 'eval_metric': 'RMSE', 'random_seed': 42, 'verbose': 0, 'bagging_temperature': 0.2, 'od_type': 'Iter', 'od_wait': 20}

pipeline_lgbm = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', LGBMRegressor(**lgbm_params))])
pipeline_xgb = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', XGBRegressor(**xgb_params))])
pipeline_cat = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', CatBoostRegressor(**cat_params))])

print("\nTreinando o modelo LightGBM...")
pipeline_lgbm.fit(X, y_log)
print("Treinando o modelo XGBoost...")
pipeline_xgb.fit(X, y_log)
print("Treinando o modelo CatBoost...")
pipeline_cat.fit(X, y_log)

print("\n? Bloco 5 conclu?do: Modelos treinados com sucesso!")

## 5. VALIDA??O E RESULTADOS

A valida??o de um modelo ? essencial para estimar sua performance em dados n?o vistos e garantir que ele n?o est? sofrendo de overfitting. A t?cnica de **Valida??o Cruzada (Cross-Validation)** foi a principal ferramenta utilizada durante o desenvolvimento para avaliar e comparar diferentes abordagens de engenharia de atributos e ajustes de hiperpar?metros. Esse processo iterativo permitiu refinar o modelo at? a vers?o final aqui apresentada.

Uma vez satisfeito com os resultados da valida??o cruzada, o modelo final foi treinado em **todo o conjunto de treinamento** para maximizar o aprendizado. As previs?es foram ent?o geradas para o conjunto de teste e submetidas ao Kaggle para a avalia??o final.

O resultado obtido na plataforma Kaggle foi um score **RMSLE de 0.2475**, um valor altamente competitivo que reflete a efic?cia das t?cnicas empregadas.

In [ ]:
# ==============================================================================
# BLOCO 6: ENSEMBLE, PREVIS?ES E SUBMISS?O
# ==============================================================================
print("\nGerando previs?es com cada modelo...")
preds_lgbm = pipeline_lgbm.predict(X_test)
preds_xgb = pipeline_xgb.predict(X_test)
preds_cat = pipeline_cat.predict(X_test)

# --- Ensemble (M?dia Ponderada) ---
final_log_predictions = 0.4 * preds_lgbm + 0.4 * preds_xgb + 0.2 * preds_cat

# Revertendo a transforma??o logar?tmica para obter o pre?o real
final_predictions = np.expm1(final_log_predictions)
final_predictions[final_predictions < 0] = 0

# --- Criando o DataFrame para submiss?o ---
submission_df = pd.DataFrame({'Id': test_ids, 'preco': final_predictions})
submission_df.to_csv('submission_final.csv', index=False)

print("\n? Bloco 6 conclu?do: Arquivo 'submission_final.csv' criado com sucesso!")
print("\nAmostra do arquivo de submiss?o:")
print(submission_df.head())

## 6. CONCLUS?O

Este trabalho demonstrou com sucesso a constru??o de um pipeline de ponta a ponta para a previs?o de pre?os de im?veis. A combina??o de uma limpeza de dados cuidadosa, uma engenharia de atributos inteligente ? com destaque para o Target Encoding manual baseado no pre?o m?dio por bairro ? e o uso de um ensemble de modelos de Gradient Boosting robustos, se provou uma estrat?gia altamente eficaz.

O score final de **0.2475** no Kaggle valida a abordagem adotada e posiciona o modelo desenvolvido como uma ferramenta de alta performance para a tarefa proposta. O processo iterativo de an?lise, modelagem e valida??o foi fundamental para alcan?ar este resultado.